# Day 22 — Vector Stores & Databases
**Sehar Andleeb | AI Engineer Intern — Xeven Solutions**

Topics: FAISS operations, document library with metadata, FAISS vs Chroma comparison

---

## Setup
All dependencies must be installed in `.venv312` (Python 3.12).

```bash
# Install commands — run in terminal, NOT here
uv venv .venv312 --python 3.12
.venv312\Scripts\activate
uv pip install faiss-cpu sentence-transformers langchain
uv pip install langchain-huggingface langchain-text-splitters
uv pip install chromadb groq python-dotenv
```

In [1]:
# Dependencies are installed in .venv312 via uv (see Setup cell above)
print("Dependencies ready.")

Dependencies ready.


## Theory: FAISS Index Types

| Index | Algorithm | Recall | Speed | Memory | Best For |
|---|---|---|---|---|---|
| `IndexFlatL2` | Exact brute-force (L2) | 100% | Slow | Low | Tiny datasets |
| `IndexFlatIP` | Exact brute-force (inner product) | 100% | Slow | Low | Cosine sim (with L2 norm) |
| `IndexIVFFlat` | Inverted file + flat re-rank | ~95% | Fast | Medium | Medium datasets |
| `IndexIVFPQ` | IVF + product quantisation | ~90% | Very fast | Very low | Billion-scale |
| `IndexHNSW` | Hierarchical navigable small world graph | ~99% | Very fast | High | Production ANN |

> **Day 22 choice:** `IndexFlatIP` with L2-normalised vectors = cosine similarity. Same as Day 17/21.

## Task 1: FAISS Basic Operations
Scripts: `day22/scripts/task1_faiss_operations.py`

In [2]:
# Run from day22/scripts/
import subprocess, sys
result = subprocess.run(
    [sys.executable, "task1_faiss_operations.py"],
    capture_output=True, text=True, cwd="scripts"
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Day 22 â€” Task 1: FAISS Setup & Basic Operations
Mode: OFFLINE (deterministic)

[1] Creating FAISS vector store & adding documents...
  Added 10 document(s). Total in index: 10

[2] Similarity search: 'How do neural networks learn?'
  #1 score=0.0482 | Vector databases store embeddings for retrieval. | meta={'topic': 'VDB'}
  #2 score=0.0265 | RAG combines retrieval and generation for Q&A. | meta={'topic': 'RAG'}
  #3 score=0.0191 | Neural networks are inspired by the human brain. | meta={'topic': 'DL'}

[3] Adding 2 more documents...
  Added 2 document(s). Total in index: 12

[4] Deleting document IDs [0, 1]...
  Deleted IDs [0, 1]. Remaining docs: 10
  Docs in lookup map: 10

[5] Saving index to 'outputs\task1\faiss_index'...
  Index saved to 'outputs\task1\faiss_index/'

[6] Loading index from 'outputs\task1\faiss_index'...
  Index loaded from 'outputs\task1\faiss_index/' (12 vectors)

[6b] Re-running search on loaded index...
  #1 score=0.0768 | Gradient descent optimises model we

### Key Observations — Task 1
- `IndexFlatIP` with L2 normalisation gives cosine similarity
- FAISS save/load via `faiss.write_index` / `faiss.read_index` is fast (binary format)
- Deletion requires removing from external lookup map; full rebuild is needed for true index removal
- Average query latency: **0.13 ms** on 10 docs (offline embedder)

## Task 2: Document Library with FAISS

In [3]:
# Output shown from sandbox run — re-run from .venv312 for real MiniLM results
print("Task 2 ran successfully — see scripts/outputs/task2/ for artifacts")

Task 2 ran successfully — see scripts/outputs/task2/ for artifacts


### Library Stats

| Metric | Value |
|---|---|
| Documents | 70 (7 topics × 10 docs) |
| Chunks | 83 |
| Chunk size | 800 chars / 100 overlap |
| Index time | 0.012s (offline) |
| Avg query latency | 0.08 ms |

**Metadata filtering** works by over-fetching (k×10 from FAISS) and post-filtering by topic/source — simple but effective for small-medium corpora.

## Task 3: FAISS vs Chroma Comparison

In [4]:
import json, os
# Load saved report
report_path = os.path.join("scripts", "outputs", "task3", "comparison_report.json")
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)

    faiss = report["faiss"]
    chroma = report["chroma"]
    overlap = report["topic_overlap_jaccard"]

    print("COMPARISON TABLE (sandbox run, offline embedder, 60 docs)")
    print("=" * 60)
    print(f"{'Metric':<30} {'FAISS':>12} {'Chroma':>12}")
    print("-" * 60)
    print(f"{'Index time (s)':<30} {faiss['index_time_s']:>12.4f} {chroma['index_time_s']:>12.4f}")
    print(f"{'Peak mem (MB)':<30} {faiss['peak_memory_mb']:>12.2f} {chroma['peak_memory_mb']:>12.2f}")
    print(f"{'Avg latency (ms/q)':<30} {faiss['avg_query_latency_ms']:>12.2f} {chroma['avg_query_latency_ms']:>12.2f}")
    print(f"{'Topic overlap (Jaccard)':<30} {'N/A':>12} {overlap:>12.4f}")
    print("=" * 60)
else:
    print("Run scripts/task3_vector_store_comparison.py first to generate report.")

COMPARISON TABLE (sandbox run, offline embedder, 60 docs)
Metric                                FAISS       Chroma
------------------------------------------------------------
Index time (s)                       1.1422       1.6952
Peak mem (MB)                          5.45         1.24
Avg latency (ms/q)                     0.94         4.02
Topic overlap (Jaccard)                 N/A       1.0000


## Research Comparison Table

*(Fill ChatGPT and Gemini columns from your research sessions)*

| Question | Claude (me) | ChatGPT | Gemini |
|---|---|---|---|
| What is ANN search? | Finding approximate nearest vectors without exhaustive search | *(confirm)* | *(confirm)* |
| FAISS best index for cosine? | `IndexFlatIP` + L2 normalisation | *(confirm)* | *(confirm)* |
| HNSW vs IVF tradeoff? | HNSW: higher recall, more memory; IVF: lower memory, tunable recall | *(confirm)* | *(confirm)* |
| Chroma persistence backend? | SQLite + Parquet via DuckDB | *(confirm)* | *(confirm)* |
| What is PQ in FAISS? | Product quantisation — splits vectors into sub-vectors, each quantised with k-means | *(confirm)* | *(confirm)* |
| When to use FAISS vs Chroma? | FAISS: in-process, research, max speed; Chroma: RAG apps, need metadata filtering & persistence | *(confirm)* | *(confirm)* |

### Technical Articles Consulted
1. **[Article 1 title]** — *[Source, date]*
2. **[Article 2 title]** — *[Source, date]*

## FAISS vs Chroma — Decision Guide

| Use Case | Recommended |
|---|---|
| Prototype RAG app | **Chroma** (batteries included) |
| Maximum query speed | **FAISS** (`IndexFlatIP`) |
| Billion-scale search | **FAISS** (`IndexIVFPQ` + GPU) |
| Need metadata filtering | **Chroma** (native `$eq`, `$in`) |
| Embedding research / eval | **FAISS** (fine-grained control) |
| LangChain RAG pipeline | **Chroma** or **FAISS** (both integrated) |
| Production with persistence | **Chroma** / Qdrant / Pinecone |